In [10]:
!pip install akshare -i https://mirrors.aliyun.com/pypi/simple/
!pip install scikit-learn -i https://mirrors.aliyun.com/pypi/simple/

Looking in indexes: https://mirrors.aliyun.com/pypi/simple/
Looking in indexes: https://mirrors.aliyun.com/pypi/simple/
     ---------------------------------------- 0.0/11.2 MB ? eta -:--:--
     ---- ----------------------------------- 1.3/11.2 MB 6.1 MB/s eta 0:00:02
     ----------- ---------------------------- 3.1/11.2 MB 7.7 MB/s eta 0:00:02
     ------------------ --------------------- 5.2/11.2 MB 8.6 MB/s eta 0:00:01
     ----------------------- ---------------- 6.6/11.2 MB 7.9 MB/s eta 0:00:01
     ----------------------------- ---------- 8.1/11.2 MB 7.9 MB/s eta 0:00:01
     ---------------------------------- ----- 9.7/11.2 MB 7.7 MB/s eta 0:00:01
     ---------------------------------------- 11.2/11.2 MB 7.9 MB/s  0:00:01
     ---------------------------------------- 0.0/46.2 MB ? eta -:--:--
     - -------------------------------------- 1.8/46.2 MB 8.4 MB/s eta 0:00:06
     --- ------------------------------------ 3.9/46.2 MB 9.4 MB/s eta 0:00:05
     ----- ----------------

In [1]:
import akshare as ak
print(ak.__version__)


1.18.13


In [3]:
# fetch_stock_price
import akshare as ak
import pandas as pd

# 1. 股票代码（贵州茅台）
stock_code = "600519"

# 2. 获取日线行情（前复权）
df = ak.stock_zh_a_hist(
    symbol=stock_code,
    period="daily",
    start_date="20180101",
    end_date="20251231",
    adjust="qfq"
)

# 3. 只保留必要字段，并统一英文命名
df = df.rename(columns={
    "日期": "date",
    "开盘": "open",
    "收盘": "close",
    "最高": "high",
    "最低": "low",
    "成交量": "volume"
})

df = df[["date", "open", "close", "high", "low", "volume"]]

# 4. 转换日期格式
df["date"] = pd.to_datetime(df["date"])

# 5. 保存为 CSV
df.to_csv("stock_price_600519.csv", index=False, encoding="utf-8-sig")

print("日线行情数据已保存，共 {} 条记录".format(len(df)))
print(df.head())


日线行情数据已保存，共 1942 条记录
        date    open   close    high     low  volume
0 2018-01-02  443.23  447.08  453.39  433.12   49612
1 2018-01-03  444.73  459.09  464.63  442.97   52019
2 2018-01-04  464.63  480.30  486.73  462.56   72205
3 2018-01-05  484.23  481.59  489.26  471.45   39989
4 2018-01-08  478.25  495.36  499.73  478.25   52205


In [4]:
# build_events
import pandas as pd

# 读取行情数据
df = pd.read_csv("stock_price_600519.csv", parse_dates=["date"])

# 每 20 个交易日构造一个事件
event_indices = df.index[::20]

events = pd.DataFrame({
    "event_date": df.loc[event_indices, "date"].values,
    "event_type": 1  # 先统一为 1
})

events.to_csv("events.csv", index=False, encoding="utf-8-sig")
print("事件表生成完成，共 {} 个事件".format(len(events)))
print(events.head())


事件表生成完成，共 98 个事件
  event_date  event_type
0 2018-01-02           1
1 2018-01-30           1
2 2018-03-06           1
3 2018-04-03           1
4 2018-05-07           1


In [5]:
# build_event_windows
import pandas as pd

price = pd.read_csv("stock_price_600519.csv", parse_dates=["date"])
events = pd.read_csv("events.csv", parse_dates=["event_date"])

price = price.sort_values("date").reset_index(drop=True)

windows = []

for _, row in events.iterrows():
    t0 = row["event_date"]

    if t0 not in price["date"].values:
        continue

    idx = price.index[price["date"] == t0][0]

    if idx < 1 or idx + 3 >= len(price):
        continue

    window = price.iloc[idx-1: idx+4].copy()
    window["event_date"] = t0
    windows.append(window)

event_windows = pd.concat(windows)
event_windows.to_csv("event_windows.csv", index=False, encoding="utf-8-sig")

print("事件窗口构造完成")
print(event_windows.head())


事件窗口构造完成
         date    open   close    high     low  volume event_date
19 2018-01-29  521.99  479.55  522.90  478.23   74536 2018-01-30
20 2018-01-30  479.23  485.31  492.00  473.25   51911 2018-01-30
21 2018-01-31  482.23  507.77  514.94  482.23   60402 2018-01-30
22 2018-02-01  510.23  500.96  510.53  496.15   50583 2018-01-30
23 2018-02-02  495.45  483.63  496.46  468.23   75421 2018-01-30


In [6]:
# build_features
import pandas as pd
import numpy as np

df = pd.read_csv("event_windows.csv", parse_dates=["date", "event_date"])

features = []

for event_date, group in df.groupby("event_date"):
    group = group.sort_values("date")

    close = group["close"].values
    volume = group["volume"].values

    returns = np.diff(close) / close[:-1]

    feat = {
        "event_date": event_date,
        "pre_return": returns[0],
        "post_1d_return": returns[1],
        "post_3d_return": (close[-1] - close[1]) / close[1],
        "volatility": returns.std(),
        "volume_change": (volume[-1] - volume[1]) / volume[1]
    }

    features.append(feat)

feature_df = pd.DataFrame(features)
feature_df.to_csv("features.csv", index=False, encoding="utf-8-sig")

print("特征构造完成")
print(feature_df.head())


特征构造完成
  event_date  pre_return  post_1d_return  post_3d_return  volatility  \
0 2018-01-30    0.012011        0.046280       -0.003462    0.030151   
1 2018-03-06   -0.003993       -0.017812        0.021539    0.017151   
2 2018-04-03   -0.005079        0.038230        0.071496    0.015912   
3 2018-05-07    0.086924        0.024150        0.036294    0.036479   
4 2018-06-04    0.075480        0.011577       -0.001904    0.033816   

   volume_change  
0       0.452891  
1      -0.532740  
2      -0.203624  
3      -0.540910  
4      -0.469311  


In [7]:
# 构造预测标签 y
feature_df["label"] = (feature_df["post_3d_return"] > 0).astype(int)
feature_df.to_csv("dataset.csv", index=False, encoding="utf-8-sig")

print("数据集构造完成")
print(feature_df["label"].value_counts())


数据集构造完成
label
0    50
1    46
Name: count, dtype: int64


In [8]:
# prepare_lstm_data
import pandas as pd
import numpy as np

# 读取事件窗口
df = pd.read_csv("event_windows.csv", parse_dates=["date", "event_date"])

X = []
y = []

for event_date, group in df.groupby("event_date"):
    group = group.sort_values("date")

    close = group["close"].values
    volume = group["volume"].values

    # 计算收益率
    returns = np.zeros_like(close)
    returns[1:] = (close[1:] - close[:-1]) / close[:-1]

    # 构造 [T, F]
    seq = np.stack([close, returns, volume], axis=1)

    # 标签：事件后 3 日累计收益
    label = int((close[-1] - close[1]) / close[1] > 0)

    X.append(seq)
    y.append(label)

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)  # (N, 5, 3)
print("y shape:", y.shape)

np.save("X_lstm.npy", X)
np.save("y_lstm.npy", y)

print("LSTM 数据准备完成")

X shape: (96, 5, 3)
y shape: (96,)
LSTM 数据准备完成


In [11]:
# normalize_lstm_data
import numpy as np
from sklearn.preprocessing import StandardScaler

X = np.load("X_lstm.npy")

N, T, F = X.shape

scaler = StandardScaler()

X_reshaped = X.reshape(-1, F)
X_scaled = scaler.fit_transform(X_reshaped)
X_scaled = X_scaled.reshape(N, T, F)

np.save("X_lstm_scaled.npy", X_scaled)

print("LSTM 输入标准化完成")

LSTM 输入标准化完成


In [13]:
!pip install tensorflow -i https://mirrors.aliyun.com/pypi/simple/

Looking in indexes: https://mirrors.aliyun.com/pypi/simple/
     ---------------------------------------- 0.0/331.7 MB ? eta -:--:--
     ---------------------------------------- 1.6/331.7 MB 9.3 MB/s eta 0:00:36
     ---------------------------------------- 3.4/331.7 MB 9.6 MB/s eta 0:00:35
      --------------------------------------- 5.0/331.7 MB 7.9 MB/s eta 0:00:42
      --------------------------------------- 6.8/331.7 MB 8.7 MB/s eta 0:00:38
     - -------------------------------------- 8.9/331.7 MB 8.5 MB/s eta 0:00:38
     - ------------------------------------- 11.0/331.7 MB 8.8 MB/s eta 0:00:37
     - ------------------------------------- 12.6/331.7 MB 8.7 MB/s eta 0:00:37
     - ------------------------------------- 15.2/331.7 MB 9.2 MB/s eta 0:00:35
     -- ------------------------------------ 17.6/331.7 MB 9.5 MB/s eta 0:00:34
     -- ------------------------------------ 19.1/331.7 MB 9.2 MB/s eta 0:00:34
     -- ------------------------------------ 20.2/331.7 MB 8.9 MB/s

In [14]:
# train_lstm
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split

# 加载数据
X = np.load("X_lstm_scaled.npy")
y = np.load("y_lstm.npy")

# 划分训练 / 测试集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 构建 LSTM 模型
model = Sequential([
    LSTM(32, input_shape=(X.shape[1], X.shape[2])),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# 训练
history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

# 测试
loss, acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {acc:.4f}")

E:\new\anaconda\envs\event_stock\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.4911 - loss: 0.6934 - val_accuracy: 0.3125 - val_loss: 0.7152
Epoch 2/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5974 - loss: 0.6774 - val_accuracy: 0.3125 - val_loss: 0.7087
Epoch 3/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6332 - loss: 0.6742 - val_accuracy: 0.3125 - val_loss: 0.7016
Epoch 4/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7754 - loss: 0.6533 - val_accuracy: 0.3750 - val_loss: 0.6952
Epoch 5/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8342 - loss: 0.6433 - val_accuracy: 0.5625 - val_loss: 0.6878
Epoch 6/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8078 - loss: 0.6297 - val_accuracy: 0.5625 - val_loss: 0.6787
Epoch 7/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7814 - loss: 0.6114 - val_accuracy: 0.5625 - val_loss: 0.6706
Epoch 8/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7852 - loss: 0.6014 - val_accuracy: 0.6875 - val_loss: 0.6635
Epoch 9